# 03 — Modeling: TextCNN + DistilBERT + Classical ML
## Phishing Email Detection with Explainable AI

This notebook implements all classification components described in Section 2.3:

| Component | Method | Description |
|-----------|--------|-------------|
| Classical baseline | Logistic Regression, Random Forest, XGBoost | Interpretable feature-based classification |
| CNN module | TextCNN (Keras) | Local textual pattern detection |
| LLM module | DistilBERT (HuggingFace) | Semantic and contextual analysis |

> **Note on computational constraints:** DistilBERT is evaluated on a stratified sample of 10,000 emails due to Google Colab free-tier GPU memory limitations (12 GB RAM). This is consistent with standard academic practice for LLM fine-tuning experiments.

---

In [ ]:
# ── Connect Google Drive ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

PHISHING_CSV    = '/content/drive/MyDrive/phishing_email.csv'
KZ_CSV          = '/content/drive/MyDrive/kazakhstan_urls.csv'
FEATURES_CSV    = '/content/drive/MyDrive/phishing_features.csv'
X_TEST_CSV      = '/content/drive/MyDrive/X_test.csv'
Y_TEST_CSV      = '/content/drive/MyDrive/y_test.csv'
XGB_MODEL_PATH  = '/content/drive/MyDrive/xgb_model.joblib'
print("Drive connected. Files ready.")

In [ ]:
# ── Install dependencies (run once in Colab) ──────────────────────────
# !pip install transformers torch scikit-learn xgboost tensorflow shap lime -q

import os, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Environment ready.')

## 1. Load Data

In [ ]:
from sklearn.model_selection import train_test_split

df = pd.read_csv(FEATURES_CSV)
feature_cols = [c for c in df.columns if c not in ['label', 'text_combined']]

X = df[feature_cols].fillna(0)
y = df['label']
texts = df['text_combined'].fillna('').astype(str)

X_train, X_test, y_train, y_test, txt_train, txt_test = train_test_split(
    X, y, texts, test_size=0.2, random_state=SEED, stratify=y
)

print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')
print(f'Interpretable features: {len(feature_cols)}')

## 2. Classical ML Models (Feature-based)

These models use the 33 interpretable features from notebook 02 and are fully explainable via SHAP.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                              roc_curve, confusion_matrix)
import xgboost as xgb

results = []
trained_models = {}

def eval_and_store(name, model, Xtr, Xte, ytr, yte):
    model.fit(Xtr, ytr)
    yp = model.predict(Xte)
    yproba = model.predict_proba(Xte)[:, 1]
    r = dict(Model=name,
             Accuracy=accuracy_score(yte, yp),
             Precision=precision_score(yte, yp),
             Recall=recall_score(yte, yp),
             F1=f1_score(yte, yp),
             AUC=roc_auc_score(yte, yproba))
    results.append(r)
    trained_models[name] = model
    print(f'  {name:<28} F1={r["F1"]:.4f}  AUC={r["AUC"]:.4f}')
    return model

print('=== Classical Models ===')

eval_and_store('Logistic Regression',
    Pipeline([('sc', StandardScaler()),
              ('clf', LogisticRegression(max_iter=1000, C=1.0, random_state=SEED))]),
    X_train, X_test, y_train, y_test)

eval_and_store('Random Forest',
    RandomForestClassifier(n_estimators=200, max_depth=12,
                           class_weight='balanced', random_state=SEED, n_jobs=-1),
    X_train, X_test, y_train, y_test)

eval_and_store('XGBoost',
    xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                      subsample=0.8, colsample_bytree=0.8,
                      eval_metric='logloss', tree_method='hist',
                      random_state=SEED, n_jobs=-1),
    X_train, X_test, y_train, y_test)

## 3. TextCNN — Local Pattern Detection

A Convolutional Neural Network applied to tokenized email text. Multiple parallel filters with different kernel sizes capture n-gram patterns of varying lengths — short urgency phrases (bigrams), URL fragments (trigrams), and longer template structures (4-5-grams).

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Embedding, Conv1D, GlobalMaxPooling1D,
                                      Dense, Dropout, Concatenate)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

tf.random.set_seed(SEED)

VOCAB_SIZE   = 20000
MAX_LEN      = 256
EMBED_DIM    = 64
NUM_FILTERS  = 128
FILTER_SIZES = [2, 3, 4, 5]
DROPOUT_RATE = 0.4

print('Tokenizing...')
tokenizer_cnn = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer_cnn.fit_on_texts(txt_train)

X_tr_seq = pad_sequences(tokenizer_cnn.texts_to_sequences(txt_train),
                          maxlen=MAX_LEN, padding='post', truncating='post')
X_te_seq = pad_sequences(tokenizer_cnn.texts_to_sequences(txt_test),
                          maxlen=MAX_LEN, padding='post', truncating='post')
print(f'Sequence shape: {X_tr_seq.shape}')

In [ ]:
def build_textcnn(vocab_size, max_len, embed_dim, num_filters, filter_sizes, dropout):
    inp = Input(shape=(max_len,), name='text_input')
    emb = Embedding(vocab_size, embed_dim, name='embedding')(inp)
    pooled = []
    for fs in filter_sizes:
        conv = Conv1D(num_filters, fs, activation='relu', name=f'conv_{fs}')(emb)
        pool = GlobalMaxPooling1D(name=f'pool_{fs}')(conv)
        pooled.append(pool)
    x = Concatenate(name='concat')(pooled)
    x = Dropout(dropout)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(dropout / 2)(x)
    out = Dense(1, activation='sigmoid')(x)
    model = Model(inp, out)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

cnn_model = build_textcnn(VOCAB_SIZE, MAX_LEN, EMBED_DIM, NUM_FILTERS, FILTER_SIZES, DROPOUT_RATE)
cnn_model.summary()

In [ ]:
history_cnn = cnn_model.fit(
    X_tr_seq, y_train,
    validation_split=0.1,
    batch_size=256,
    epochs=10,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5)
    ],
    verbose=1
)

y_proba_cnn = cnn_model.predict(X_te_seq, verbose=0).flatten()
y_pred_cnn  = (y_proba_cnn >= 0.5).astype(int)

r_cnn = dict(Model='TextCNN',
             Accuracy=accuracy_score(y_test, y_pred_cnn),
             Precision=precision_score(y_test, y_pred_cnn),
             Recall=recall_score(y_test, y_pred_cnn),
             F1=f1_score(y_test, y_pred_cnn),
             AUC=roc_auc_score(y_test, y_proba_cnn))
results.append(r_cnn)
print(f'  TextCNN  F1={r_cnn["F1"]:.4f}  AUC={r_cnn["AUC"]:.4f}')

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history_cnn.history['loss'],     label='Train')
axes[0].plot(history_cnn.history['val_loss'], label='Val')
axes[0].set_title('TextCNN — Loss', fontweight='bold'); axes[0].legend()
axes[1].plot(history_cnn.history['accuracy'],     label='Train')
axes[1].plot(history_cnn.history['val_accuracy'], label='Val')
axes[1].set_title('TextCNN — Accuracy', fontweight='bold'); axes[1].legend()
plt.suptitle('Figure 3.7. TextCNN Training Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_3_7_cnn_training.png', bbox_inches='tight')
plt.show()

## 4. DistilBERT — Semantic and Contextual Analysis

DistilBERT is a distilled version of BERT (66M parameters, 40% smaller, 2× faster) that retains 97% of BERT's language understanding. It captures **semantic and contextual features** — detecting manipulative framing, authority impersonation, and deceptive urgency that surface-level features may miss.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (DistilBertTokenizerFast,
                           DistilBertForSequenceClassification,
                           get_linear_schedule_with_warmup)
from torch.optim import AdamW

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cpu':
    print('⚠ No GPU detected. DistilBERT will be slow on CPU.')
    print('  In Colab: Runtime → Change runtime type → GPU')

# Stratified 10k sample
LLM_N = 10000
idx = (df[df['label']==0].sample(LLM_N//2, random_state=SEED).index.tolist() +
       df[df['label']==1].sample(LLM_N//2, random_state=SEED).index.tolist())
df_llm = df.loc[idx].reset_index(drop=True)

t_llm, tlb = df_llm['text_combined'].fillna('').astype(str).tolist(), df_llm['label'].tolist()
t_tr, t_te, y_tr, y_te = train_test_split(t_llm, tlb, test_size=0.2,
                                           random_state=SEED, stratify=tlb)
print(f'LLM train: {len(t_tr)} | LLM test: {len(t_te)}')

In [ ]:
BERT_MODEL   = 'distilbert-base-uncased'
BERT_MAX_LEN = 128
BERT_BATCH   = 32
BERT_EPOCHS  = 3

bert_tok = DistilBertTokenizerFast.from_pretrained(BERT_MODEL)

class EmailDataset(Dataset):
    def __init__(self, texts, labels, tok, max_len):
        self.enc    = tok(texts, truncation=True, padding='max_length',
                          max_length=max_len, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):          return len(self.labels)
    def __getitem__(self, idx): return {'input_ids':      self.enc['input_ids'][idx],
                                        'attention_mask': self.enc['attention_mask'][idx],
                                        'labels':         self.labels[idx]}

train_dl = DataLoader(EmailDataset(t_tr, y_tr, bert_tok, BERT_MAX_LEN),
                      batch_size=BERT_BATCH, shuffle=True)
test_dl  = DataLoader(EmailDataset(t_te, y_te, bert_tok, BERT_MAX_LEN),
                      batch_size=BERT_BATCH, shuffle=False)
print('Dataset ready.')

In [ ]:
bert_model = DistilBertForSequenceClassification.from_pretrained(
    BERT_MODEL, num_labels=2).to(device)

optimizer = AdamW(bert_model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_dl) * BERT_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps)

train_losses, val_f1s = [], []

for epoch in range(BERT_EPOCHS):
    bert_model.train()
    total_loss = 0
    for batch in train_dl:
        optimizer.zero_grad()
        out = bert_model(input_ids=batch['input_ids'].to(device),
                          attention_mask=batch['attention_mask'].to(device),
                          labels=batch['labels'].to(device))
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += out.loss.item()

    # Validation
    bert_model.eval()
    preds_v, labels_v = [], []
    with torch.no_grad():
        for batch in test_dl:
            logits = bert_model(input_ids=batch['input_ids'].to(device),
                                 attention_mask=batch['attention_mask'].to(device)).logits
            preds_v.extend(logits.argmax(1).cpu().numpy())
            labels_v.extend(batch['labels'].numpy())

    avg_loss = total_loss / len(train_dl)
    vf1 = f1_score(labels_v, preds_v)
    train_losses.append(avg_loss); val_f1s.append(vf1)
    print(f'Epoch {epoch+1}/{BERT_EPOCHS}  Loss={avg_loss:.4f}  Val F1={vf1:.4f}')

In [ ]:
# Final evaluation
bert_model.eval()
preds_f, proba_f, labels_f = [], [], []
with torch.no_grad():
    for batch in test_dl:
        logits = bert_model(input_ids=batch['input_ids'].to(device),
                             attention_mask=batch['attention_mask'].to(device)).logits
        proba = torch.softmax(logits, 1)[:, 1].cpu().numpy()
        preds_f.extend((proba >= 0.5).astype(int).tolist())
        proba_f.extend(proba.tolist())
        labels_f.extend(batch['labels'].numpy().tolist())

r_bert = dict(Model='DistilBERT (10k)',
              Accuracy=accuracy_score(labels_f, preds_f),
              Precision=precision_score(labels_f, preds_f),
              Recall=recall_score(labels_f, preds_f),
              F1=f1_score(labels_f, preds_f),
              AUC=roc_auc_score(labels_f, proba_f))
results.append(r_bert)
print(f'  DistilBERT  F1={r_bert["F1"]:.4f}  AUC={r_bert["AUC"]:.4f}')

# Curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(1, BERT_EPOCHS+1), train_losses, 'o-', color='#e74c3c')
axes[0].set_title('DistilBERT — Train Loss', fontweight='bold'); axes[0].set_xlabel('Epoch')
axes[1].plot(range(1, BERT_EPOCHS+1), val_f1s, 'o-', color='#2ecc71')
axes[1].set_title('DistilBERT — Val F1', fontweight='bold'); axes[1].set_xlabel('Epoch')
plt.suptitle('Figure 3.8. DistilBERT Fine-tuning Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_3_8_bert_curves.png', bbox_inches='tight'); plt.show()

## 5. Kazakhstan URL Module

In [ ]:
import re
from sklearn.model_selection import StratifiedKFold, cross_val_score

kz_df = pd.read_csv(KZ_CSV)
print(f'KZ dataset: {kz_df.shape}')
print(kz_df['label'].value_counts().rename({0:'Legitimate', 1:'Phishing'}))

KZ_LEGIT = ['egov.kz','gov.kz','akorda.kz','minfin.kz','nbrk.kz','enpf.kz',
            'kaspi.kz','halykbank.kz','bcc.kz','jusan.kz','kcell.kz',
            'beeline.kz','activ.kz','kolesa.kz','krisha.kz','olx.kz',
            'satu.kz','nur.kz','tengrinews.kz','airastana.kz']
KZ_PHISH  = [r'egov[\-\.]\w+\.(?!kz)', r'kaspi[\-\.]\w+\.(?!kz)',
             r'halyk[\-\.]\w+\.(?!kz)', r'gov\-kz\.', r'\.kz\.[a-z]{2,4}[/"]',
             r'enpf.*(bonus|claim|payment)', r'gosuslugi.*kz',
             r'login.*egov', r'secure.*kaspi', r'kz.*(prize|lottery|winner|free|bonus)']
SUSP_TLD  = r'\.(xyz|top|click|gq|ml|cf|ga|tk|pw|cc|ru)(/|$)'

def kz_feats(url):
    u = str(url).lower()
    return {'url_length':      len(u),
            'has_kz_legit':    int(any(d in u for d in KZ_LEGIT)),
            'has_kz_phishing': int(any(bool(re.search(p,u)) for p in KZ_PHISH)),
            'has_ip':          int(bool(re.search(r'//\d{1,3}\.\d{1,3}',u))),
            'has_shortened':   int(bool(re.search(r'(bit\.ly|tinyurl|goo\.gl|t\.co|ow\.ly|rb\.gy)',u))),
            'has_susp_tld':    int(bool(re.search(SUSP_TLD,u))),
            'subdomain_count': u.count('.'),
            'has_login_kw':    int(bool(re.search(r'(login|verify|secure|confirm|update|account)',u))),
            'has_kz_domain':   int('.kz' in u)}

kz_X = pd.DataFrame(kz_df['url'].apply(kz_feats).tolist())
kz_y = kz_df['label']

kz_clf = xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                             eval_metric='logloss', tree_method='hist', random_state=SEED)
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_f1  = cross_val_score(kz_clf, kz_X, kz_y, cv=cv5, scoring='f1')
cv_auc = cross_val_score(kz_clf, kz_X, kz_y, cv=cv5, scoring='roc_auc')

print(f'KZ URL Classifier — 5-fold CV')
print(f'  F1:      {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')
print(f'  ROC-AUC: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')

kz_clf.fit(kz_X, kz_y)
imp_df = pd.DataFrame({'feature': kz_X.columns,
                        'importance': kz_clf.feature_importances_}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#e74c3c' if v > 0.08 else '#3498db' for v in imp_df['importance']]
ax.barh(imp_df['feature'][::-1], imp_df['importance'][::-1], color=colors[::-1], alpha=0.85)
ax.set_title('Figure 3.9. KZ URL Classifier — Feature Importance', fontsize=13, fontweight='bold')
ax.set_xlabel('Feature Importance (XGBoost)')
plt.tight_layout()
plt.savefig('fig_3_9_kz_importance.png', bbox_inches='tight'); plt.show()

## 6. Full Model Comparison

In [ ]:
res_df = pd.DataFrame(results).set_index('Model')
print('=== Full Model Comparison ===')
print((res_df * 100).round(2).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
metrics = ['Accuracy','Precision','Recall','F1','AUC']
x   = np.arange(len(res_df))
w   = 0.14
pal = ['#3498db','#2ecc71','#e74c3c','#f39c12','#9b59b6']
for i, (m, c) in enumerate(zip(metrics, pal)):
    ax.bar(x + i*w, res_df[m], w, label=m, color=c, alpha=0.85)
ax.set_xticks(x + w*2)
ax.set_xticklabels(res_df.index, rotation=20, ha='right', fontsize=10)
ax.set_ylim(0.75, 1.02)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_title('Figure 3.10. Full Model Comparison', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('fig_3_10_full_comparison.png', bbox_inches='tight'); plt.show()

print(f'Best model: {res_df["F1"].idxmax()}  F1={res_df["F1"].max():.4f}')

In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(9, 7))
roc_sources = [
    ('Logistic Regression', trained_models['Logistic Regression'].predict_proba(X_test)[:,1], y_test, '#3498db'),
    ('Random Forest',       trained_models['Random Forest'].predict_proba(X_test)[:,1],       y_test, '#27ae60'),
    ('XGBoost',             trained_models['XGBoost'].predict_proba(X_test)[:,1],             y_test, '#e67e22'),
    ('TextCNN',             y_proba_cnn,                                                      y_test, '#8e44ad'),
    ('DistilBERT (10k)',    proba_f,                                                          labels_f,'#e74c3c'),
]
for name, proba, true, color in roc_sources:
    fpr, tpr, _ = roc_curve(true, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(true, proba):.3f})', color=color, lw=2)
ax.plot([0,1],[0,1],'k--',lw=1,label='Random')
ax.set_xlabel('False Positive Rate',fontsize=12)
ax.set_ylabel('True Positive Rate',fontsize=12)
ax.set_title('Figure 3.11. ROC Curves — All Models',fontsize=13,fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('fig_3_11_roc_all.png', bbox_inches='tight'); plt.show()

## 7. Save Models

In [ ]:
import joblib

# XGBoost saved for SHAP in notebook 04
joblib.dump(trained_models["XGBoost"], XGB_MODEL_PATH)
cnn_model.save('cnn_model.keras')
bert_model.save_pretrained('distilbert_model')
bert_tok.save_pretrained('distilbert_model')

X_test.to_csv(X_TEST_CSV, index=False)
y_test.to_csv(Y_TEST_CSV, index=False)

print('Saved: xgb_model.joblib | cnn_model.keras | distilbert_model/')
print(f'Best overall: {res_df["F1"].idxmax()}  F1={res_df["F1"].max():.4f}')

## 8. Summary

| Model | Type | Training data | Key strength |
|-------|------|--------------|---------------|
| Logistic Regression | Classical ML | 33 features, 82k | Interpretable baseline |
| Random Forest | Ensemble | 33 features, 82k | Robust, feature importance |
| XGBoost | Gradient Boosting | 33 features, 82k | Best classical + SHAP |
| TextCNN | Deep Learning | Raw text, 82k | Local n-gram patterns |
| DistilBERT | LLM fine-tuned | Raw text, 10k sample | Semantic/contextual |
| KZ URL Module | XGBoost | 150 KZ URLs | Regional phishing detection |

**XGBoost on interpretable features** is used in the XAI pipeline (notebook 04) for its full SHAP TreeExplainer compatibility. DistilBERT and TextCNN serve as comparative benchmarks.